# Démo 1 — Tokens, contexte et coût

**Contexte :** assistant du service client de Lagune Banque.

Cette démo présente la tokenisation, le coût des tokens d'entrée et de sortie, et l'effet de l'historique sur la taille du contexte.

**Sommaire**
- [Configuration](#configuration)
- [Tokenisation](#tokenisation)
- [Coût des tokens](#cout-des-tokens)
- [Historique de conversation](#historique)
- [Questions regroupées](#questions-regroupees)
- [À retenir](#a-retenir)

## <a id="configuration"></a>Configuration

In [1]:
import json
import os
import re
import sys
import time
from pathlib import Path

import litellm
import pandas as pd

sys.path.insert(0, str(next(p for p in [Path("code_partage"), Path("../code_partage"), Path("../../code_partage"), Path("../../../code_partage"), Path("../../../../code_partage")] if (p / "config.py").exists()).resolve()))
from config import prepare

In [2]:
prepare()

Connexion OK : gemini/gemini-3.5-flash-lite


In [3]:
SEED = 42


def ask(messages, **options):
    debut = time.perf_counter()
    r = litellm.completion(model=options.pop("model", os.environ["MODELE"]), api_key=os.environ["LLM_API_KEY"], seed=SEED, num_retries=4, messages=messages, **options)
    return {"text": (r.choices[0].message.content or "").strip(), "in": r.usage.prompt_tokens, "out": r.usage.completion_tokens, "seconds": time.perf_counter() - debut, "raw": r}

## <a id="tokenisation"></a>Tokenisation

`litellm.token_counter` mesure le nombre de tokens d'un texte pour le modèle sélectionné. Comparons une même idée formulée de quatre façons.

In [4]:
TEXTES = {
    "français": "Bonjour, je voudrais connaître le plafond de mon Prêt Express.",
    "anglais": "Hello, I would like to know the limit of my Express Loan.",
    "SMS": "slt jvx savoir cmb coute le pret expres stp",
    "chiffres et références": "25 000 FCFA le 02/10/2026 VIR-48213",
}
pd.DataFrame([{"texte": nom, "caractères": len(t), "mots": len(t.split()), "tokens": litellm.token_counter(model=os.environ["MODELE"], text=t),
               "tokens par mot": round(litellm.token_counter(model=os.environ["MODELE"], text=t) / len(t.split()), 1)} for nom, t in TEXTES.items()]).set_index("texte")

,caractères,mots,tokens,tokens par mot
texte,,,,
français,62,10,18,1.8
anglais,57,12,14,1.2
SMS,43,9,12,1.3
chiffres et références,35,6,17,2.8


Le nombre de tokens ne correspond pas directement au nombre de mots. La langue, les chiffres et les références modifient la quantité de tokens nécessaire.

## <a id="cout-des-tokens"></a>Coût des tokens

Les tarifs proviennent de LiteLLM et sont exprimés en dollars par million de tokens.

In [5]:
prix_in = litellm.cost_per_token(model=os.environ["MODELE"], prompt_tokens=1_000_000, completion_tokens=0)[0]
prix_out = litellm.cost_per_token(model=os.environ["MODELE"], prompt_tokens=0, completion_tokens=1_000_000)[1]
print(f"{os.environ['MODELE']} : {prix_in:.2f} $ par million de tokens en entrée, {prix_out:.2f} $ en sortie (x{prix_out / prix_in:.1f})")

gemini/gemini-3.5-flash-lite : 0.30 $ par million de tokens en entrée, 2.50 $ en sortie (x8.3)


## <a id="historique"></a>Historique de conversation

À chaque tour, l'application renvoie au modèle l'ensemble des messages précédents, puis ajoute la nouvelle question. Le modèle ne conserve pas de mémoire entre deux appels. Les colonnes ci-dessous montrent comment l'historique s'allonge.

<table style="width:100%; border:0; border-collapse:separate; border-spacing:10px 0; color:#202124; font-size:0.72em; line-height:1.05; zoom:0.9;">
  <tr>
    <td style="width:25%; vertical-align:top; border:0; padding:0 3px;">
      <div style="font-weight:700; margin-bottom:5px; color:#202124;">Tour 1<br>2 messages</div>
      <div style="padding:6px; border-radius:5px; background:#e8eaf6; color:#202124;"><strong>SYSTEM</strong><br>Consignes</div>
      <div style="height:8px; line-height:8px;">&nbsp;</div>
      <div style="padding:6px; border-radius:5px; background:#e3f2fd; color:#202124;"><strong>USER</strong><br>Question 1</div>
    </td>
    <td style="width:25%; vertical-align:top; border:0; padding:0 3px;">
      <div style="font-weight:700; margin-bottom:5px; color:#202124;">Tour 2<br>4 messages</div>
      <div style="padding:6px; border-radius:5px; background:#e8eaf6; color:#202124;"><strong>SYSTEM</strong><br>Consignes</div>
      <div style="height:8px; line-height:8px;">&nbsp;</div>
      <div style="padding:6px; border-radius:5px; background:#e3f2fd; color:#202124;"><strong>USER</strong><br>Question 1</div>
      <div style="height:8px; line-height:8px;">&nbsp;</div>
      <div style="padding:6px; border-radius:5px; background:#e8f5e9; color:#202124;"><strong>ASSISTANT</strong><br>Réponse 1</div>
      <div style="height:8px; line-height:8px;">&nbsp;</div>
      <div style="padding:6px; border-radius:5px; background:#e3f2fd; color:#202124;"><strong>USER</strong><br>Question 2</div>
    </td>
    <td style="width:25%; vertical-align:top; border:0; padding:0 3px;">
      <div style="font-weight:700; margin-bottom:5px; color:#202124;">Tour 3<br>6 messages</div>
      <div style="padding:6px; border-radius:5px; background:#e8eaf6; color:#202124;"><strong>SYSTEM</strong><br>Consignes</div>
      <div style="height:8px; line-height:8px;">&nbsp;</div>
      <div style="padding:6px; border-radius:5px; background:#e3f2fd; color:#202124;"><strong>USER</strong><br>Question 1</div>
      <div style="height:8px; line-height:8px;">&nbsp;</div>
      <div style="padding:6px; border-radius:5px; background:#e8f5e9; color:#202124;"><strong>ASSISTANT</strong><br>Réponse 1</div>
      <div style="height:8px; line-height:8px;">&nbsp;</div>
      <div style="padding:6px; border-radius:5px; background:#e3f2fd; color:#202124;"><strong>USER</strong><br>Question 2</div>
      <div style="height:8px; line-height:8px;">&nbsp;</div>
      <div style="padding:6px; border-radius:5px; background:#e8f5e9; color:#202124;"><strong>ASSISTANT</strong><br>Réponse 2</div>
      <div style="height:8px; line-height:8px;">&nbsp;</div>
      <div style="padding:6px; border-radius:5px; background:#e3f2fd; color:#202124;"><strong>USER</strong><br>Question 3</div>
    </td>
    <td style="width:25%; vertical-align:top; border:0; padding:0 3px;">
      <div style="font-weight:700; margin-bottom:5px; color:#202124;">Tour 4<br>8 messages</div>
      <div style="padding:6px; border-radius:5px; background:#e8eaf6; color:#202124;"><strong>SYSTEM</strong><br>Consignes</div>
      <div style="height:8px; line-height:8px;">&nbsp;</div>
      <div style="padding:6px; border-radius:5px; background:#e3f2fd; color:#202124;"><strong>USER</strong><br>Question 1</div>
      <div style="height:8px; line-height:8px;">&nbsp;</div>
      <div style="padding:6px; border-radius:5px; background:#e8f5e9; color:#202124;"><strong>ASSISTANT</strong><br>Réponse 1</div>
      <div style="height:8px; line-height:8px;">&nbsp;</div>
      <div style="padding:6px; border-radius:5px; background:#e3f2fd; color:#202124;"><strong>USER</strong><br>Question 2</div>
      <div style="height:8px; line-height:8px;">&nbsp;</div>
      <div style="padding:6px; border-radius:5px; background:#e8f5e9; color:#202124;"><strong>ASSISTANT</strong><br>Réponse 2</div>
      <div style="height:8px; line-height:8px;">&nbsp;</div>
      <div style="padding:6px; border-radius:5px; background:#e3f2fd; color:#202124;"><strong>USER</strong><br>Question 3</div>
      <div style="height:8px; line-height:8px;">&nbsp;</div>
      <div style="padding:6px; border-radius:5px; background:#e8f5e9; color:#202124;"><strong>ASSISTANT</strong><br>Réponse 3</div>
      <div style="height:8px; line-height:8px;">&nbsp;</div>
      <div style="padding:6px; border-radius:5px; background:#e3f2fd; color:#202124;"><strong>USER</strong><br>Question 4</div>
    </td>
  </tr>
</table>

La réponse de l'assistant au tour 1 est ajoutée à l'historique après sa génération ; elle est donc renvoyée avec la question suivante.

In [6]:
QUESTIONS = [
    "Bonjour, quels sont les frais de retard du Prêt Express ?",
    "Et pour le Prêt Express+ ?",
    "Comment fait-on pour rembourser en avance ?",
    "Que se passe-t-il si j'ai un impayé ?",
    "Combien de temps avant de pouvoir emprunter à nouveau ?",
    "Et si mon KYC est incomplet ?",
    "Quelles sont les heures d'ouverture du service client ?",
    "Merci, résume-moi les trois points importants de notre échange.",
]
SYSTEME = {"role": "system", "content": "Tu es l'assistant du service client de Lagune Banque. Réponds en deux phrases au plus."}

historique, tours = [SYSTEME], []
for i, question in enumerate(QUESTIONS, 1):
    historique.append({"role": "user", "content": question})
    r = ask(historique, max_tokens=150)
    historique.append({"role": "assistant", "content": r["text"]})
    tours.append({"tour": i, "tokens lus (entrée)": r["in"], "tokens écrits (sortie)": r["out"]})
conversation = pd.DataFrame(tours).set_index("tour")
conversation["entrée cumulée"] = conversation["tokens lus (entrée)"].cumsum()
conversation

,tokens lus (entrée),tokens écrits (sortie),entrée cumulée
tour,,,
1,35,59,35
2,104,50,139
3,166,45,305
4,229,51,534
5,294,43,828
6,347,45,1175
7,406,41,1581
8,463,61,2044


## <a id="questions-regroupees"></a>Questions regroupées

Comparons les mêmes huit questions envoyées en un seul message, sans renvoyer l'historique à chaque tour.

In [7]:
groupe = ask([SYSTEME, {"role": "user", "content": "Réponds à chaque question en deux phrases au plus.\n" + "\n".join(f"{i}. {q}" for i, q in enumerate(QUESTIONS, 1))}], max_tokens=800)
entree_conv, sortie_conv = int(conversation["tokens lus (entrée)"].sum()), int(conversation["tokens écrits (sortie)"].sum())


def cout(tokens_in, tokens_out):
    return tokens_in * prix_in / 1e6 + tokens_out * prix_out / 1e6


pd.DataFrame([
    {"façon de procéder": "huit tours de conversation", "tokens d'entrée": entree_conv, "tokens de sortie": sortie_conv, "coût en $": round(cout(entree_conv, sortie_conv), 5)},
    {"façon de procéder": "un seul message groupé", "tokens d'entrée": groupe["in"], "tokens de sortie": groupe["out"], "coût en $": round(cout(groupe["in"], groupe["out"]), 5)},
]).set_index("façon de procéder")

,tokens d'entrée,tokens de sortie,coût en $
façon de procéder,,,
huit tours de conversation,2044,395,0.00160
un seul message groupé,148,69,0.00022


In [8]:
premier, dernier = conversation["tokens lus (entrée)"].iloc[0], conversation["tokens lus (entrée)"].iloc[-1]
print(f"Au tour 8, le modèle relit {dernier} tokens, contre {premier} au tour 1 (x{dernier / premier:.1f}).")
print(f"La conversation envoie {entree_conv} tokens en entrée au total, le message groupé {groupe['in']} (x{entree_conv / groupe['in']:.1f}).")

Au tour 8, le modèle relit 463 tokens, contre 35 au tour 1 (x13.2).
La conversation envoie 2044 tokens en entrée au total, le message groupé 148 (x13.8).


## <a id="a-retenir"></a>À retenir

- La facturation dépend des tokens, pas du nombre de mots.
- Les tokens de sortie coûtent plus cher que les tokens d'entrée.
- À chaque tour, l'historique est renvoyé au modèle et augmente le contexte transmis.
- Regrouper les questions, démarrer une nouvelle conversation ou résumer l'historique peut réduire le contexte.
- La fenêtre de contexte inclut les messages, les documents, les définitions d'outils et la réponse générée.